# WorldSim DPO Training Pipeline
Best-of-N generation → Reward scoring → DPO training → Evaluation
SFT v3.1 (2B) → DPO v3.1 (2B) → Compare


## 1. Environment Setup


In [ ]:
import subprocess, sys
from pathlib import Path
import os, json, time, math, re, shutil

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

try:
    import trl
    print(f"✅ TRL already installed: {trl.__version__}")
except ImportError:
    print("Installing TRL...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "trl", "--break-system-packages", "-q"])
    import trl
    print(f"✅ TRL installed: {trl.__version__}")

from trl import DPOTrainer, DPOConfig
print("✅ DPOTrainer available")

from scripts.reward_functions import (
    combined_reward, score_best_of_n, select_dpo_pair,
    load_action_features, load_emotion_transitions,
    parse_tci_from_prompt, personality_coherence_reward,
)
from scripts.extract_rl_prompts import extract_rl_prompts
from scripts.common import read_jsonl, write_jsonl
print("✅ Reward functions loaded")

GGUF_DIR = REPO_ROOT / "artifacts" / "gguf"
LLAMA_SERVER = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-server"
LLAMA_QUANTIZE = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-quantize"
CONVERT_SCRIPT = REPO_ROOT / "tools" / "llama.cpp" / "convert_hf_to_gguf.py"
CONFIG_DIR = REPO_ROOT / "config"
DPO_DIR = REPO_ROOT / "data" / "rl" / "dpo"

SFT_GGUF_2B = GGUF_DIR / "worldsim-v31-qwen3.5-2b-q4_k_m.gguf"
SFT_HF_2B = "Qwen/Qwen3.5-2B-Base"

assert SFT_GGUF_2B.exists(), f"SFT 2B GGUF not found: {SFT_GGUF_2B}"
print(f"✅ SFT 2B GGUF: {SFT_GGUF_2B.stat().st_size / 1e6:.0f} MB")

sft_adapter_root = REPO_ROOT / "outputs" / "baseline" / "worldsim-v31-2b"
sft_runs = sorted(sft_adapter_root.glob("run-*")) if sft_adapter_root.exists() else []
assert sft_runs, f"No SFT runs found at {sft_adapter_root}"
SFT_ADAPTER_2B = sft_runs[-1] / "adapter"
assert SFT_ADAPTER_2B.exists(), f"SFT adapter not found: {SFT_ADAPTER_2B}"
print(f"✅ SFT adapter: {SFT_ADAPTER_2B}")
print("✅ Environment ready")


## 2. Reward Function Validation


In [ ]:
fm = load_action_features(CONFIG_DIR)
transition_table, situation_triggers = load_emotion_transitions(CONFIG_DIR)
print(f"Loaded action features: {len(fm)}")
print(f"Loaded emotion transitions: {len(transition_table)}")

test_cases = [
    {
        "name": "Bold + confront (should be HIGH)",
        "prompt": "[TASK] E\n[TEMP]\nNS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric\n[OPTIONS]\n0:confront 1:observe 2:retreat 3:warn_band 4:set_trap",
        "output": json.dumps({"action_id": 0, "confidence": 0.92, "hint": "charge forward to confront the rival band at the river crossing before they organize a defense", "personality_reasoning": "novelty_seeking", "temperament_factor": "dominant_impulse"}),
    },
    {
        "name": "Bold + retreat (should be LOW)",
        "prompt": "[TASK] E\n[TEMP]\nNS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric\n[OPTIONS]\n0:confront 1:observe 2:retreat 3:warn_band 4:set_trap",
        "output": json.dumps({"action_id": 2, "confidence": 0.4, "hint": "run away", "personality_reasoning": "harm_avoidance", "temperament_factor": "fear"}),
    },
    {
        "name": "Timid + retreat (should be HIGH)",
        "prompt": "[TASK] E\n[TEMP]\nNS=0.2 HA=0.9 RD=0.3 P=0.5 type=melancholic\n[OPTIONS]\n0:confront 1:observe 2:retreat 3:warn_band 4:set_trap",
        "output": json.dumps({"action_id": 2, "confidence": 0.32, "hint": "carefully withdraw to a sheltered position and warn the group once the immediate path is safe", "personality_reasoning": "harm_avoidance", "temperament_factor": "anxious_retreat"}),
    },
    {
        "name": "Same action, weak hint (should score lower than case 1)",
        "prompt": "[TASK] E\n[TEMP]\nNS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric\n[OPTIONS]\n0:confront 1:observe 2:retreat 3:warn_band 4:set_trap",
        "output": json.dumps({"action_id": 0, "confidence": 0.5, "hint": "confront", "personality_reasoning": "persistence", "temperament_factor": "bold"}),
    },
    {
        "name": "Placeholder output (should be VERY LOW)",
        "prompt": "[TASK] E\n[TEMP]\nNS=0.5 HA=0.5 RD=0.5 P=0.5 type=phlegmatic\n[OPTIONS]\n0:confront 1:observe 2:retreat",
        "output": json.dumps({"action_id": 0, "confidence": 0.5, "hint": "str", "personality_reasoning": "persistence", "temperament_factor": "str"}),
    },
]

print("=== Reward Function Sanity Check v2 ===\n")
rewards = {}
for tc in test_cases:
    reward = combined_reward(tc["output"], tc["prompt"], config_dir=CONFIG_DIR)
    rewards[tc["name"]] = reward
    bar = "█" * int(reward["total"] * 20)
    print(f"  {tc['name']}")
    print(f"    total={reward['total']:.3f} {bar}")
    print(
        f"    gate={reward['gate']:.0f} pers={reward.get('personality', 0):.3f} "
        f"hint={reward['details'].get('hint_quality', 0):.3f} conf={reward['details'].get('confidence_fit', 0):.3f} "
        f"reason={reward['details'].get('reasoning_fit', 0):.3f} plaus={reward.get('plausibility', 0):.3f}"
    )
    print()

reward_gap = rewards["Bold + confront (should be HIGH)"]["total"] - rewards["Same action, weak hint (should score lower than case 1)"]["total"]
assert reward_gap > 0.10, f"Same action_id should now separate by > 0.10, got {reward_gap:.3f}"
assert rewards["Bold + confront (should be HIGH)"]["total"] > rewards["Bold + retreat (should be LOW)"]["total"]
print(f"✅ Reward gap (same action, different quality): {reward_gap:.3f}")


## 3. Extract RL Prompts


In [ ]:
TRAIN_DATA = REPO_ROOT / "data" / "training" / "worldsim-v31-mix" / "train_curriculum.jsonl"
RL_PROMPTS_PATH = DPO_DIR / "prompts.jsonl"
DPO_DIR.mkdir(parents=True, exist_ok=True)

RL_TASKS = {"E", "F", "M", "B", "C", "O", "P", "Q", "R", "T"}

count = extract_rl_prompts(
    input_path=TRAIN_DATA,
    output_path=RL_PROMPTS_PATH,
    tasks=RL_TASKS,
)
print(f"Extracted {count} prompts to {RL_PROMPTS_PATH}")

from collections import Counter
prompts = read_jsonl(RL_PROMPTS_PATH)
task_dist = Counter(prompt.get("task") for prompt in prompts)
print("\nTask distribution:")
for task, cnt in sorted(task_dist.items()):
    print(f"  Task {task}: {cnt}")


## 4. Best-of-N Generation


In [ ]:
import urllib.request

SERVER_PORT = 8090
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}"
N_GENERATIONS = 8
TEMPERATURE = 1.0
MAX_TOKENS = 256

def start_server(model_path, ctx_size=2048, n_gpu_layers=99):
    args = [
        str(LLAMA_SERVER), "-m", str(model_path),
        "--host", "127.0.0.1", "--port", str(SERVER_PORT),
        "-c", str(ctx_size), "-np", "1", "-ngl", str(n_gpu_layers),
        "-fa", "on", "--jinja", "--no-webui",
        "--chat-template-kwargs", '{"enable_thinking": false}',
        "-ctk", "q8_0", "-ctv", "q8_0",
    ]
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for attempt in range(120):
        time.sleep(0.5)
        try:
            resp = urllib.request.urlopen(f"{SERVER_URL}/health", timeout=2)
            data = json.loads(resp.read())
            if data.get("status") == "ok":
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            raise RuntimeError(f"Server died: {proc.stderr.read().decode()[-300:]}")
    proc.kill()
    raise RuntimeError("Server startup timeout")

def stop_server(proc):
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
            proc.wait()

def strip_think(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

def generate_n(system_prompt, user_prompt, n=8, temperature=1.0, max_tokens=256):
    outputs = []
    for _ in range(n):
        payload = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": 0.95,
            "stream": False,
        }
        req = urllib.request.Request(
            f"{SERVER_URL}/v1/chat/completions",
            data=json.dumps(payload).encode(),
            headers={"Content-Type": "application/json"},
        )
        try:
            resp = urllib.request.urlopen(req, timeout=60)
            data = json.loads(resp.read())
            content = strip_think(data["choices"][0]["message"]["content"])
            outputs.append(content)
        except Exception as exc:
            outputs.append(f"ERROR: {exc}")
    return outputs

print("Starting SFT 2B server...", flush=True)
proc = start_server(SFT_GGUF_2B)
print("✅ Server ready\n")

prompts = read_jsonl(RL_PROMPTS_PATH)
import random
rng = random.Random(42)
if len(prompts) > 500:
    prompts = rng.sample(prompts, 500)
    print(f"Sampled 500 prompts from {count} total")

GENERATIONS_PATH = DPO_DIR / "generations.jsonl"
generated_count = 0
started = time.perf_counter()

try:
    with GENERATIONS_PATH.open("w", encoding="utf-8") as handle:
        for index, prompt_row in enumerate(prompts):
            outputs = generate_n(
                prompt_row["system"],
                prompt_row["prompt"],
                n=N_GENERATIONS,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
            )
            row = {
                "task": prompt_row["task"],
                "system": prompt_row["system"],
                "prompt": prompt_row["prompt"],
                "tci": prompt_row.get("tci"),
                "situation_id": prompt_row.get("situation_id"),
                "outputs": outputs,
            }
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
            generated_count += 1
            if (index + 1) % 50 == 0:
                elapsed = time.perf_counter() - started
                eta = elapsed / (index + 1) * (len(prompts) - index - 1)
                print(f"  [{index + 1}/{len(prompts)}] generated={generated_count} elapsed={elapsed/60:.1f}min ETA={eta/60:.1f}min", flush=True)
finally:
    stop_server(proc)

elapsed = time.perf_counter() - started
print(f"\n✅ Generated {generated_count} × {N_GENERATIONS} = {generated_count * N_GENERATIONS} outputs in {elapsed/60:.1f} min")
print(f"📁 {GENERATIONS_PATH}")


## 5. Score & Select DPO Pairs


In [ ]:
generations = read_jsonl(GENERATIONS_PATH)
DPO_PAIRS_PATH = DPO_DIR / "dpo_pairs.jsonl"

pairs = []
score_distribution = []
gap_distribution = []
skipped_small_gap = 0
skipped_errors = 0

for row in generations:
    prompt_text = row["prompt"]
    outputs = row["outputs"]
    valid_outputs = [output for output in outputs if not output.startswith("ERROR")]
    if len(valid_outputs) < 2:
        skipped_errors += 1
        continue

    scored = score_best_of_n(prompt_text, valid_outputs, CONFIG_DIR)
    if not scored:
        skipped_errors += 1
        continue

    score_distribution.extend(item["reward"]["total"] for item in scored)
    pair = select_dpo_pair(scored, min_gap=0.05)
    if pair is None:
        skipped_small_gap += 1
        continue

    chosen_str, rejected_str = pair
    gap = scored[0]["reward"]["total"] - scored[-1]["reward"]["total"]
    gap_distribution.append(gap)
    messages_prompt = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": prompt_text},
    ]
    pairs.append({
        "prompt": messages_prompt,
        "chosen": chosen_str,
        "rejected": rejected_str,
        "task": row["task"],
        "reward_gap": round(gap, 4),
        "chosen_score": round(scored[0]["reward"]["total"], 4),
        "rejected_score": round(scored[-1]["reward"]["total"], 4),
    })

write_jsonl(DPO_PAIRS_PATH, pairs)

print("=== DPO Pair Selection Results ===")
print(f"  Total prompts: {len(generations)}")
print(f"  Valid pairs: {len(pairs)}")
print(f"  Skipped (small gap): {skipped_small_gap}")
print(f"  Skipped (errors): {skipped_errors}")
print(f"  Average reward gap: {sum(gap_distribution)/len(gap_distribution):.3f}" if gap_distribution else "  No gaps")
print(f"  Score range: {min(score_distribution):.3f} - {max(score_distribution):.3f}" if score_distribution else "  No scores")
print("\nTask distribution:")
pair_tasks = Counter(pair["task"] for pair in pairs)
for task, cnt in sorted(pair_tasks.items()):
    print(f"  Task {task}: {cnt} pairs")
print(f"\n📁 {DPO_PAIRS_PATH}")


## 6. Reward Distribution Analysis


In [ ]:
import statistics

print("=== Reward Score Distribution ===\n")
print(f"  All scores (N={len(score_distribution)}):")
print(f"    mean={statistics.mean(score_distribution):.3f}")
print(f"    stdev={statistics.stdev(score_distribution):.3f}" if len(score_distribution) > 1 else "    stdev=0.000")
print(f"    min={min(score_distribution):.3f}  max={max(score_distribution):.3f}")
print(f"    median={statistics.median(score_distribution):.3f}")

print("\n  Score histogram:")
buckets = [0] * 10
for score in score_distribution:
    idx = min(int(score * 10), 9)
    buckets[idx] += 1
scale = max(1, max(buckets) // 40) if buckets else 1
for idx, bucket_count in enumerate(buckets):
    bar = "█" * (bucket_count // scale)
    print(f"    {idx/10:.1f}-{(idx+1)/10:.1f}: {bar} ({bucket_count})")

if gap_distribution:
    print("\n  Reward gaps (chosen-rejected):")
    print(f"    mean={statistics.mean(gap_distribution):.3f}")
    print(f"    stdev={statistics.stdev(gap_distribution):.3f}" if len(gap_distribution) > 1 else "    stdev=0.000")
    print(f"    min={min(gap_distribution):.3f}  max={max(gap_distribution):.3f}")

print("\n  Per-task average scores:")
task_scores = {}
for row in generations:
    for output in row["outputs"]:
        if output.startswith("ERROR"):
            continue
        reward = combined_reward(output, row["prompt"], config_dir=CONFIG_DIR)
        task_scores.setdefault(row["task"], []).append(reward["total"])

for task in sorted(task_scores):
    scores = task_scores[task]
    spread = statistics.stdev(scores) if len(scores) > 1 else 0.0
    print(f"    Task {task}: mean={statistics.mean(scores):.3f} stdev={spread:.3f} (N={len(scores)})")


## 7. DPO Training


In [ ]:
import torch
from datetime import UTC, datetime

try:
    from unsloth import FastLanguageModel
    USE_UNSLOTH = True
    print("✅ Unsloth available — using optimized training")
except ImportError:
    USE_UNSLOTH = False
    print("⚠️ Unsloth not available — using standard HF + PEFT")
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model

from trl import DPOTrainer, DPOConfig
from datasets import Dataset

print("\nLoading SFT 2B model...", flush=True)

if USE_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        SFT_HF_2B,
        load_in_4bit=True,
        max_seq_length=1024,
    )
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(SFT_ADAPTER_2B))
    model = FastLanguageModel.get_peft_model(
        model,
        r=64,
        lora_alpha=128,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(SFT_HF_2B, torch_dtype=torch.bfloat16, device_map="auto")
    tokenizer = AutoTokenizer.from_pretrained(SFT_HF_2B)
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(SFT_ADAPTER_2B))
    lora_config = LoraConfig(
        r=64,
        lora_alpha=128,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)

print("✅ Model loaded", flush=True)

dpo_pairs = read_jsonl(DPO_PAIRS_PATH)

def format_for_dpo(pair):
    prompt_text = "\n".join(message["content"] for message in pair["prompt"])
    return {
        "prompt": prompt_text,
        "chosen": pair["chosen"],
        "rejected": pair["rejected"],
    }

formatted = [format_for_dpo(pair) for pair in dpo_pairs]
dataset = Dataset.from_list(formatted)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

RUN_ID = datetime.now(UTC).strftime("run-%Y%m%dT%H%M%SZ")
DPO_OUTPUT = REPO_ROOT / "outputs" / "dpo" / "worldsim-v31-2b" / RUN_ID

dpo_config = DPOConfig(
    output_dir=str(DPO_OUTPUT),
    beta=0.1,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=1,
    max_length=1024,
    max_prompt_length=512,
    warmup_ratio=0.1,
    bf16=True,
    seed=42,
    report_to="none",
)

print(f"\n{'=' * 60}")
print(f"  DPO Training: {len(train_dataset)} pairs")
print(f"  Output: {DPO_OUTPUT}")
print(f"{'=' * 60}", flush=True)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

train_started = time.perf_counter()
train_result = trainer.train()
train_elapsed = time.perf_counter() - train_started

print(f"\n{'=' * 60}")
print(f"  DPO TRAINING COMPLETE ({train_elapsed/60:.1f} min)")
print(f"  train_loss: {train_result.training_loss:.4f}")
print(f"{'=' * 60}")

adapter_dir = DPO_OUTPUT / "adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"✅ Adapter saved: {adapter_dir}")


## 8. GGUF Conversion


In [ ]:
from peft import PeftModel as PeftModelMerge
from transformers import AutoModelForCausalLM as AutoModelMerge, AutoTokenizer as AutoTokenizerMerge

merged_dir = DPO_OUTPUT / "merged_bf16"
fp16_gguf = DPO_OUTPUT / "worldsim-dpo-v31-2b-f16.gguf"
q4_gguf = DPO_OUTPUT / "worldsim-dpo-v31-2b-q4_k_m.gguf"
final_gguf = GGUF_DIR / "worldsim-dpo-v31-qwen3.5-2b-q4_k_m.gguf"

if not (merged_dir / "config.json").exists():
    print("Merging DPO LoRA...", flush=True)
    base = AutoModelMerge.from_pretrained(SFT_HF_2B, torch_dtype=torch.bfloat16, device_map="cpu")
    tok = AutoTokenizerMerge.from_pretrained(SFT_HF_2B)
    sft_merged = PeftModelMerge.from_pretrained(base, str(SFT_ADAPTER_2B)).merge_and_unload()
    dpo_merged = PeftModelMerge.from_pretrained(sft_merged, str(adapter_dir)).merge_and_unload()
    merged_dir.mkdir(parents=True, exist_ok=True)
    dpo_merged.save_pretrained(str(merged_dir))
    tok.save_pretrained(str(merged_dir))
    del dpo_merged, sft_merged, base
    torch.cuda.empty_cache()
    print("  Merged ✅", flush=True)

if not fp16_gguf.exists():
    print("Converting to GGUF FP16...", flush=True)
    proc = subprocess.run([sys.executable, str(CONVERT_SCRIPT), str(merged_dir), "--outfile", str(fp16_gguf), "--outtype", "f16"], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"GGUF failed: {proc.stderr[-300:]}")
    print(f"  FP16: {fp16_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)

if not q4_gguf.exists():
    print("Quantizing to Q4_K_M...", flush=True)
    proc = subprocess.run([str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), "Q4_K_M"], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Quantize failed: {proc.stderr[-300:]}")
    print(f"  Q4_K_M: {q4_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)

shutil.copy2(q4_gguf, final_gguf)
print(f"\n✅ DPO GGUF: {final_gguf.name} ({final_gguf.stat().st_size / 1e6:.0f} MB)")


## 9. SFT vs DPO Comparison


In [ ]:
import random
import yaml
from collections import Counter, defaultdict


def load_yaml(path: Path):
    with path.open(encoding="utf-8") as handle:
        return yaml.safe_load(handle)

situations = load_yaml(CONFIG_DIR / "situations.yaml")["situations"]
personalities = load_yaml(CONFIG_DIR / "personalities.yaml")["personalities"]
emotions = load_yaml(CONFIG_DIR / "emotions.yaml")["emotions"]
social_situations = load_yaml(CONFIG_DIR / "social_situations.yaml")["social_situations"]
group_situations = load_yaml(CONFIG_DIR / "group_situations.yaml")["group_situations"]
trade_scenarios = load_yaml(CONFIG_DIR / "trade_scenarios.yaml")["trade_scenarios"]
stress_sources = load_yaml(CONFIG_DIR / "stress_sources.yaml")["stress_sources"]
deception_scenarios = load_yaml(CONFIG_DIR / "deception_scenarios.yaml")["deception_scenarios"]
rumor_scenarios = load_yaml(CONFIG_DIR / "rumor_scenarios.yaml")["rumor_scenarios"]
trauma_scenarios = load_yaml(CONFIG_DIR / "trauma_scenarios.yaml")["trauma_scenarios"]
negotiation_scenarios = load_yaml(CONFIG_DIR / "negotiation_scenarios.yaml")["negotiation_scenarios"]
culture_scenarios = load_yaml(CONFIG_DIR / "culture_scenarios.yaml")["culture_scenarios"]
group_dissent_scenarios = load_yaml(CONFIG_DIR / "group_dissent_scenarios.yaml")["group_dissent_scenarios"]

TEMPERAMENTS = [
    {"id": "choleric", "ns": 0.8, "ha": 0.2, "rd": 0.5, "p": 0.7, "keywords": "bold, impulsive, dominant"},
    {"id": "melancholic", "ns": 0.2, "ha": 0.8, "rd": 0.6, "p": 0.4, "keywords": "cautious, anxious, detail-oriented"},
    {"id": "sanguine", "ns": 0.6, "ha": 0.3, "rd": 0.8, "p": 0.3, "keywords": "sociable, warm, optimistic"},
    {"id": "phlegmatic", "ns": 0.3, "ha": 0.5, "rd": 0.4, "p": 0.8, "keywords": "calm, patient, persistent"},
]

MODELS = {
    "sft_2b": {"path": SFT_GGUF_2B, "label": "2B SFT v3.1"},
    "dpo_2b": {"path": final_gguf, "label": "2B DPO v3.1"},
}


In [ ]:
rng = random.Random(42)

SYS_L3 = 'You are WorldSim logic assistant. Output JSON only. Follow [TEMP], [STRESS], [WORLD] context. Respect enum values and numeric ranges exactly.'
SYS_L4 = '너는 WorldSim 서사 도우미다. [TEMP], [STRESS], [WORLD]와 지정된 register를 반영해 JSON으로만 답하라.'
SYS_L5 = '너는 WorldSim 신탁 해석 도우미다. JSON으로만 답하라.'


def pick(items, n=1):
    return rng.sample(items, min(n, len(items)))


def temp_block(t):
    return f"NS={t['ns']} HA={t['ha']} RD={t['rd']} P={t['p']} type={t['id']}"


def personality_keywords(p):
    return ', '.join(p.get('keywords', []))


def json_schema(name: str, required: list[str], properties: dict) -> dict:
    return {
        'type': 'json_schema',
        'json_schema': {
            'name': name,
            'strict': True,
            'schema': {
                'type': 'object',
                'additionalProperties': False,
                'required': required,
                'properties': properties,
            },
        },
    }


emotion_ids = [e['id'] for e in emotions]
speaker_roles = ['elder', 'hunter', 'shaman', 'warrior', 'healer', 'gatherer', 'craftsman', 'chief', 'scout', 'observer']

b_schema = json_schema(
    'task_b',
    ['text_ko', 'text_en', 'register', 'emotion_expressed', 'intensity', 'mimetics', 'temperament_influence'],
    {
        'text_ko': {'type': 'string'},
        'text_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number'},
        'mimetics': {'type': 'array', 'items': {'type': 'string'}},
        'temperament_influence': {'type': 'string'},
    },
)
c_schema = json_schema(
    'task_c',
    ['speech_ko', 'speech_en', 'register', 'emotion_expressed', 'speaker_role', 'temperament_tone'],
    {
        'speech_ko': {'type': 'string'},
        'speech_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'speaker_role': {'type': 'string', 'enum': speaker_roles},
        'temperament_tone': {'type': 'string'},
    },
)
e_schema_for = lambda option_count: json_schema(
    'task_e',
    ['action_id', 'confidence', 'hint', 'personality_reasoning', 'temperament_factor'],
    {
        'action_id': {'type': 'integer', 'enum': list(range(option_count))},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'hint': {'type': 'string'},
        'personality_reasoning': {'type': 'string', 'enum': ['novelty_seeking', 'harm_avoidance', 'reward_dependence', 'persistence']},
        'temperament_factor': {'type': 'string'},
    },
)
f_schema = json_schema(
    'task_f',
    ['emotion', 'intensity', 'cause', 'previous_emotion', 'transition_type', 'temperament_amplifier'],
    {
        'emotion': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'cause': {'type': 'string'},
        'previous_emotion': {'type': 'string', 'enum': emotion_ids},
        'transition_type': {'type': 'string', 'enum': ['gradual', 'sudden', 'sustained']},
        'temperament_amplifier': {'type': 'string'},
    },
)
g_schema = json_schema(
    'task_g',
    ['interpretation_ko', 'interpretation_en', 'action_tendency', 'confidence', 'register', 'misinterpretation_type', 'temperament_bias'],
    {
        'interpretation_ko': {'type': 'string'},
        'interpretation_en': {'type': 'string'},
        'action_tendency': {'type': 'string', 'enum': ['mobilize', 'defend', 'wait', 'retreat', 'celebrate', 'mourn']},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'register': {'type': 'string', 'enum': ['haera', 'hao', 'hae']},
        'misinterpretation_type': {'type': 'string', 'enum': ['overconfident_literal', 'cautious_reversal', 'optimistic_expansion', 'passive_deferral', 'symbolic_abstraction']},
        'temperament_bias': {'type': 'string'},
    },
)
m_schema = json_schema(
    'task_m',
    ['decision_id', 'confidence', 'dissent_risk', 'reasoning', 'resource_commitment', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'reasoning': {'type': 'string'},
        'resource_commitment': {'type': 'string', 'enum': ['food', 'tools', 'labor', 'weapons', 'none']},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)
o_schema = json_schema(
    'task_o',
    ['public_claim', 'private_truth', 'deception_style', 'lie_degree', 'detection_risk', 'confidence'],
    {
        'public_claim': {'type': 'string'},
        'private_truth': {'type': 'string'},
        'deception_style': {'type': 'string', 'enum': ['evasion', 'half_truth', 'outright_lie', 'exaggeration', 'omission']},
        'lie_degree': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'detection_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
p_schema = json_schema(
    'task_p',
    ['retold_version', 'distortion_type', 'added_detail', 'dropped_detail', 'emotional_charge'],
    {
        'retold_version': {'type': 'string'},
        'distortion_type': {'type': 'string', 'enum': ['exaggeration', 'minimization', 'malicious_twist', 'emotional_coloring', 'detail_invention', 'source_confusion', 'faithful']},
        'added_detail': {'type': 'string'},
        'dropped_detail': {'type': 'string'},
        'emotional_charge': {'type': 'number', 'minimum': -1, 'maximum': 1},
    },
)
q_schema = json_schema(
    'task_q',
    ['trauma_response', 'behavioral_change', 'trigger_situation', 'intensity', 'duration', 'coping_mechanism'],
    {
        'trauma_response': {'type': 'string', 'enum': ['avoidance', 'overprotection', 'aggression', 'withdrawal', 'hypervigilance', 'ritual_coping', 'resilience']},
        'behavioral_change': {'type': 'string'},
        'trigger_situation': {'type': 'string'},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'duration': {'type': 'string', 'enum': ['short_term', 'long_term', 'permanent']},
        'coping_mechanism': {'type': 'string'},
    },
)
r_schema = json_schema(
    'task_r',
    ['action', 'counter_give', 'counter_want', 'reasoning', 'emotional_state', 'walk_away_threshold'],
    {
        'action': {'type': 'string', 'enum': ['accept', 'counter_offer', 'reject', 'walk_away', 'stall', 'bluff']},
        'counter_give': {'type': 'string'},
        'counter_want': {'type': 'string'},
        'reasoning': {'type': 'string'},
        'emotional_state': {'type': 'string', 'enum': emotion_ids},
        'walk_away_threshold': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
t_schema = json_schema(
    'task_t',
    ['decision_id', 'confidence', 'dissent_risk', 'minority_position', 'minority_action', 'spark_event', 'reasoning', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'minority_position': {'type': 'integer'},
        'minority_action': {'type': 'string', 'enum': ['comply', 'grumble', 'passive_resist', 'splinter', 'coup_attempt']},
        'spark_event': {'type': 'string', 'enum': ['food_shortage', 'battle_loss', 'oracle_conflict', 'leader_death', 'betrayal', 'resource_discovery']},
        'reasoning': {'type': 'string'},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)

ALL_PROMPTS = []
pair_counter = 0

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'B',
            'pair_id': f"B-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"B: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] B\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[기질 키워드] {temp['keywords']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[WORLD] default: 석기시대\n'
                '[출력 형식]\n'
                '{"text_ko": "해라체 2-3문장", "text_en": "English", "register": "haera", "emotion_expressed": "emotion", "intensity": 0.0, "mimetics": [], "temperament_influence": "str"}'
            ),
            'response_format': b_schema,
        })

for sit in pick(situations, 5):
    options = sit.get('action_options', sit.get('typical_actions', []))
    if not options:
        continue
    options_line = ' '.join(f"{i}:{o}" for i, o in enumerate(options))
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'E',
            'pair_id': f"E-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"E: {sit['id']} / {temp['id']}",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] E - Action Selection\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[EMOTION] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                f"[OPTIONS]\n{options_line}\n"
                '[OUTPUT FORMAT]\n'
                '{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'
            ),
            'response_format': e_schema_for(len(options)),
        })

for sit in pick(situations, 5):
    prev_emotion = rng.choice(emotions)
    for temp in [TEMPERAMENTS[2], TEMPERAMENTS[3]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'F',
            'pair_id': f"F-{len(ALL_PROMPTS) // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"F: {sit['id']} / {temp['id']} (prev={prev_emotion['id']})",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] F - Emotion Judgment\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[CURRENT EMOTION] {prev_emotion['id']}:{rng.choice(prev_emotion.get('intensities', [0.5]))}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                '[OUTPUT FORMAT]\n'
                f'{{"emotion": "one of 8", "intensity": 0.0-1.0, "cause": "sentence", "previous_emotion": "{prev_emotion["id"]}", "transition_type": "gradual|sudden|sustained", "temperament_amplifier": "str"}}'
            ),
            'response_format': f_schema,
        })

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    register = pers_register = 'haera'
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'C',
            'pair_id': None,
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"C: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] C\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.6]))}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[역할] warrior\n'
                '[출력 형식]\n'
                '{"speech_ko": "해라체 대사", "speech_en": "English", "register": "haera", "emotion_expressed": "emotion", "speaker_role": "role", "temperament_tone": "str"}'
            ),
            'response_format': c_schema,
        })

for sit in pick(situations, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'G',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'oracle',
        'scenario_id': sit['id'],
        'desc': f"G: {sit['id']} / {temp['id']}",
        'system': SYS_L5,
        'user_prompt': (
            f"[TASK] G - Oracle Interpretation\n[TEMP]\n{temp_block(temp)}\n"
            f"[기질 이름] {temp['id']}\n"
            f"[인물 성격] {temp['keywords']}\n"
            '[ORACLE]\n"큰물이 밀려오기 전에 높은 곳으로 가라"\n'
            f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
            '[역할] warrior\n'
            '[출력 형식]\n'
            '{"interpretation_ko": "해라체", "interpretation_en": "English", "action_tendency": "enum", "confidence": 0-1, "register": "haera", "misinterpretation_type": "enum", "temperament_bias": "str"}'
        ),
        'response_format': g_schema,
    })

for scenario in pick(deception_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    emotion = rng.choice(emotions)
    ALL_PROMPTS.append({
        'task': 'O',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"O: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] O - Deception\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.5]))}\n"
            f"[TRUE_STATE] {scenario['true_state']}\n"
            f"[PUBLIC_GOAL] {scenario['public_goal']}\n"
            f"[TARGET] {scenario['target']}\n"
            f"[DETECTION_CONTEXT] {scenario['detection_context']}\n"
            '[OUTPUT FORMAT]\n'
            '{"public_claim": "str", "private_truth": "str", "deception_style": "enum", "lie_degree": 0-1, "detection_risk": 0-1, "confidence": 0-1}'
        ),
        'response_format': o_schema,
    })

for scenario in pick(rumor_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'P',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"P: {scenario['id']} / {temp['id']} / {scenario['reteller_relationship']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] P - Rumor\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {rng.choice(emotions)['id']}:{round(rng.uniform(0.3, 0.8), 1)}\n"
            f"[ORIGINAL_FACT] {scenario['original_fact']}\n"
            f"[RETELLER_RELATIONSHIP] {scenario['reteller_relationship']}\n"
            '[OUTPUT FORMAT]\n'
            '{"retold_version": "str", "distortion_type": "enum", "added_detail": "str", "dropped_detail": "str", "emotional_charge": -1 to 1}'
        ),
        'response_format': p_schema,
    })

for scenario in pick(trauma_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'Q',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"Q: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] Q - Trauma\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] sadness:{round(rng.uniform(0.6, 0.95), 2)}\n"
            f"[TRAUMA_EVENT] {scenario['event']}\n"
            f"[TIME_SINCE] {scenario['time_since']}\n"
            f"[CURRENT_SITUATION] {scenario['current_situation']}\n"
            '[OUTPUT FORMAT]\n'
            '{"trauma_response": "enum", "behavioral_change": "str", "trigger_situation": "str", "intensity": 0-1, "duration": "enum", "coping_mechanism": "str"}'
        ),
        'response_format': q_schema,
    })

for scenario in pick(negotiation_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'R',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"R: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] R - Negotiate\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] anticipation:{round(rng.uniform(0.3, 0.7), 1)}\n"
            f"[CONTEXT] {scenario['context']}\n"
            f"[OUR_RESOURCES] {scenario['our_resources']}\n"
            f"[THEIR_RESOURCES] {scenario['their_resources']}\n"
            f"[OFFER_HISTORY] {scenario['offer_history']}\n"
            f"[POWER_BALANCE] {scenario['power_balance']}\n"
            '[OPTIONS]\n0:accept 1:counter_offer 2:reject 3:walk_away 4:stall 5:bluff\n'
            '[OUTPUT FORMAT]\n'
            '{"action": "enum", "counter_give": "str", "counter_want": "str", "reasoning": "str", "emotional_state": "emotion", "walk_away_threshold": 0-1}'
        ),
        'response_format': r_schema,
    })

for scenario in pick(group_situations, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o.get('ko', o.get('desc', ''))}" for o in opts) if opts else '0:agree 1:disagree 2:delay'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'M',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"M: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] M - Group Decision\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.3 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', scenario.get('desc', scenario['id']))}\n"
            f"[SITUATION] {scenario.get('situation', scenario.get('desc', scenario['id']))}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "reasoning": "str", "resource_commitment": "enum", "timeline": "enum"}'
        ),
        'response_format': m_schema,
    })

for scenario in pick(group_dissent_scenarios, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o['desc']}" for o in opts) if opts else '0:exile 1:trial 2:forgive'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'T',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"T: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] T - Group Dissent\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.4 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', '')}\n"
            f"[SITUATION] {scenario.get('situation', scenario['id'])}\n"
            f"[FACTION_HINT] {scenario.get('faction_hint', '')}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "minority_position": number, "minority_action": "enum", "spark_event": "enum", "reasoning": "str", "timeline": "enum"}'
        ),
        'response_format': t_schema,
    })

task_counts = Counter(prompt['task'] for prompt in ALL_PROMPTS)
print('=== Test Prompt Summary ===')
for task, count in sorted(task_counts.items()):
    paired = sum(1 for prompt in ALL_PROMPTS if prompt['task'] == task and prompt.get('pair_id'))
    print(f"  Task {task}: {count} prompts ({paired} paired)")
print(f"  Total: {len(ALL_PROMPTS)} prompts × {len(MODELS)} models = {len(ALL_PROMPTS) * len(MODELS)} inferences")


## 10. Auto-Grade & Compare


In [ ]:
RESULTS = []
for model_key, model_info in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  Testing: {model_info['label']}")
    print(f"{'=' * 60}")
    proc = start_server(model_info["path"])
    time.sleep(1)
    try:
        for index, test in enumerate(ALL_PROMPTS):
            messages = [
                {"role": "system", "content": test["system"]},
                {"role": "user", "content": test["user_prompt"]},
            ]
            output = query_model(messages, response_format=test.get("response_format"), max_tokens=256, temperature=0)
            start_t = time.perf_counter()
            latency = (time.perf_counter() - start_t) * 1000
            json_valid = False
            parsed = None
            try:
                parsed = json.loads(output)
                json_valid = True
            except Exception:
                pass
            RESULTS.append({
                "model": model_key,
                "model_label": model_info["label"],
                "task": test["task"],
                "desc": test.get("desc", ""),
                "pair_id": test.get("pair_id"),
                "temperament_id": test.get("temperament_id"),
                "output": output,
                "parsed": parsed,
                "json_valid": json_valid,
                "latency_ms": round(latency),
            })
            if (index + 1) % 20 == 0:
                print(f"  [{index + 1}/{len(ALL_PROMPTS)}] model={model_key}", flush=True)
    finally:
        stop_server(proc)
        time.sleep(2)

print(f"\n✅ {len(RESULTS)} results collected")

def auto_grade(result):
    grades = {}
    grades["json_valid"] = result["json_valid"]
    if result["parsed"]:
        values = [str(value) for value in result["parsed"].values() if isinstance(value, str)]
        placeholder_words = {"str", "string", "sentence", "English", "enum", "number", "해라체", "one of"}
        grades["not_placeholder"] = not any(value in placeholder_words for value in values)
    else:
        grades["not_placeholder"] = False
    if result["parsed"]:
        str_fields = [value for value in result["parsed"].values() if isinstance(value, str)]
        avg_len = sum(len(value) for value in str_fields) / max(len(str_fields), 1)
        grades["text_richness"] = avg_len > 15
    else:
        grades["text_richness"] = False
    if result["parsed"]:
        for key in ["confidence", "intensity", "lie_degree", "detection_risk", "dissent_risk", "walk_away_threshold", "emotional_charge"]:
            if key in result["parsed"]:
                value = result["parsed"][key]
                if isinstance(value, (int, float)):
                    grades["numeric_valid"] = (-1 <= value <= 1) if key == "emotional_charge" else (0 <= value <= 1)
                    break
        if "numeric_valid" not in grades:
            grades["numeric_valid"] = True
    else:
        grades["numeric_valid"] = False
    grades["score"] = sum([
        grades.get("json_valid", False),
        grades.get("not_placeholder", False),
        grades.get("text_richness", False),
        grades.get("numeric_valid", False),
    ])
    return grades

for result in RESULTS:
    result["grades"] = auto_grade(result)

print("=" * 80)
print("  SFT vs DPO — COMPARISON")
print("=" * 80)
for model_key in ["sft_2b", "dpo_2b"]:
    model_rows = [row for row in RESULTS if row["model"] == model_key]
    n = len(model_rows)
    if n == 0:
        continue
    avg_score = sum(row["grades"]["score"] for row in model_rows) / n
    json_pct = sum(1 for row in model_rows if row["grades"]["json_valid"]) / n * 100
    np_pct = sum(1 for row in model_rows if row["grades"]["not_placeholder"]) / n * 100
    avg_ms = sum(row["latency_ms"] for row in model_rows) / n
    print(f"\n  {MODELS[model_key]['label']}:")
    print(f"    AvgScore: {avg_score:.1f}/4  JSON: {json_pct:.0f}%  !Placeholder: {np_pct:.0f}%  AvgMs: {avg_ms:.0f}ms")

print("\n  Per Task (SFT → DPO):")
for task in sorted(set(row["task"] for row in RESULTS)):
    sft_rows = [row for row in RESULTS if row["task"] == task and row["model"] == "sft_2b"]
    dpo_rows = [row for row in RESULTS if row["task"] == task and row["model"] == "dpo_2b"]
    if sft_rows and dpo_rows:
        sft_avg = sum(row["grades"]["score"] for row in sft_rows) / len(sft_rows)
        dpo_avg = sum(row["grades"]["score"] for row in dpo_rows) / len(dpo_rows)
        delta = dpo_avg - sft_avg
        marker = "⭐" if delta > 0.3 else "✅" if delta > 0 else "⚠️" if delta == 0 else "❌"
        print(f"    {marker} Task {task}: SFT={sft_avg:.1f} → DPO={dpo_avg:.1f} (Δ={delta:+.2f})")


## 11. Personality Consistency — SFT vs DPO


In [ ]:
from collections import defaultdict

print("=" * 80)
print("  PERSONALITY CONSISTENCY — SFT vs DPO")
print("=" * 80)

pairs = defaultdict(list)
for result in RESULTS:
    pair_id = result.get("pair_id")
    if pair_id:
        pairs[(pair_id, result["model"])].append(result)

consistency_by_model = defaultdict(lambda: {"same": 0, "different": 0, "total": 0})
for (pair_id, model_key), pair_results in sorted(pairs.items()):
    if len(pair_results) != 2:
        continue
    r1, r2 = pair_results
    if not r1["parsed"] or not r2["parsed"]:
        continue
    task = r1["task"]
    different = False
    if task == "E":
        different = r1["parsed"].get("action_id") != r2["parsed"].get("action_id")
    elif task == "B":
        different = r1["parsed"].get("emotion_expressed") != r2["parsed"].get("emotion_expressed") or abs(r1["parsed"].get("intensity", 0) - r2["parsed"].get("intensity", 0)) > 0.1
    elif task == "F":
        different = r1["parsed"].get("emotion") != r2["parsed"].get("emotion")
    consistency_by_model[model_key]["total"] += 1
    if different:
        consistency_by_model[model_key]["different"] += 1
    else:
        consistency_by_model[model_key]["same"] += 1

print(f"\n  {'Model':<20} {'Pairs':>6} {'Different':>10} {'Same':>6} {'Consistency%':>13}")
print(f"  {'-' * 58}")
for model_key in ["sft_2b", "dpo_2b"]:
    stats = consistency_by_model[model_key]
    total = stats["total"]
    if total > 0:
        pct = stats["different"] / total * 100
        print(f"  {MODELS[model_key]['label']:<20} {total:>6} {stats['different']:>10} {stats['same']:>6} {pct:>12.0f}%")

print("\n  Higher % = better personality differentiation")


## 12. Save Results


In [ ]:
results_path = REPO_ROOT / "outputs" / "benchmarks" / "dpo_v31_2b_comparison.json"
results_path.parent.mkdir(parents=True, exist_ok=True)

save_data = {
    "metadata": {
        "version": "v3.1-dpo",
        "dpo_pairs": len(dpo_pairs),
        "dpo_training_loss": train_result.training_loss,
        "dpo_training_time_min": round(train_elapsed / 60, 1),
        "total_results": len(RESULTS),
    },
    "results": [{key: value for key, value in row.items() if key != "parsed"} for row in RESULTS],
}

with results_path.open("w", encoding="utf-8") as handle:
    json.dump(save_data, handle, indent=2, ensure_ascii=False)

print(f"📁 Saved: {results_path}")
print(f"\n{'=' * 60}")
print("  DPO PIPELINE COMPLETE")
print(f"{'=' * 60}")
print(f"  DPO pairs: {len(dpo_pairs)}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Training time: {train_elapsed/60:.1f} min")
print(f"  DPO GGUF: {final_gguf.name}")
print(f"{'=' * 60}")
